# Lasso regression

In [ ]:
import warnings
import numpy as np
import pandas as pd
from sklearn.datasets import load_diabetes, make_regression
from sklearn.linear_model import Lasso, LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
warnings.simplefilter("ignore")

## One input: the slope reaches exactly 0

In [ ]:
X, y = make_regression(n_samples=100, n_features=1, n_informative=1, noise=20, random_state=13)
for alpha in [0, 1, 5, 10, 20, 25, 30]:
    model = LinearRegression() if alpha == 0 else Lasso(alpha=alpha)
    model.fit(X, y)
    print(f"alpha {alpha:2d}: slope {model.coef_[0]:6.2f}, intercept {model.intercept_:.2f}")
xc, yc = X.ravel() - X.mean(), y - y.mean()
print("slope is 0 from alpha =", round((xc * yc).mean(), 2))

## Degree-16 polynomial: which powers survive?

In [ ]:
rng = np.random.default_rng(42)
x = 5 * rng.random((100, 1)) - 2
y2 = 0.7 * x[:, 0] ** 2 - 2 * x[:, 0] + 3 + rng.standard_normal(100)
for alpha in [0.01, 0.1, 1]:
    model = make_pipeline(PolynomialFeatures(degree=16, include_bias=False), StandardScaler(),
                          Lasso(alpha=alpha, max_iter=200000)).fit(x, y2)
    kept = [f"x^{p + 1}" for p in np.nonzero(model[-1].coef_)[0]]
    print(f"alpha {alpha}: {len(kept)} non-zero ->", kept)

## Diabetes: coefficients and feature selection

In [ ]:
data = load_diabetes()
X_train, X_test, y_train, y_test = train_test_split(data.data, data.target, test_size=0.2, random_state=2)
rows = {}
for alpha in [0, 0.001, 0.01, 0.1, 0.5, 1, 2, 5]:
    reg = LinearRegression() if alpha == 0 else Lasso(alpha=alpha, max_iter=100000)
    reg.fit(X_train, y_train)
    rows[alpha] = list(reg.coef_.round(1)) + [int((reg.coef_ == 0).sum()), round(reg.score(X_test, y_test), 3)]
pd.DataFrame(rows, index=data.feature_names + ["zeros", "test R2"]).T

## Bias and variance against alpha (bootstrap estimate)

In [ ]:
Xtr, Xte, ytr, yte = train_test_split(x, y2, test_size=0.2, random_state=2)

def bias_variance(alpha, rounds=200):
    r = np.random.default_rng(123)
    preds = []
    for _ in range(rounds):
        i = r.integers(0, len(ytr), len(ytr))
        model = make_pipeline(PolynomialFeatures(degree=16, include_bias=False), StandardScaler(),
                              LinearRegression() if alpha == 0 else Lasso(alpha=alpha, max_iter=200000))
        preds.append(model.fit(Xtr[i], ytr[i]).predict(Xte))
    preds = np.array(preds)
    return ((preds - yte) ** 2).mean(), ((preds.mean(0) - yte) ** 2).mean(), preds.var(0).mean()

for alpha in [0, 0.001, 0.01, 0.1, 1, 3]:
    loss, bias2, var = bias_variance(alpha)
    print(f"alpha {alpha:>6}: error {loss:9.2f}  bias2 {bias2:6.2f}  variance {var:9.2f}")